# CellRank 轨迹潜能分析（AST 轨迹亚群）

**描述**: 基于 slingshot 伪时间与转录组相似性构建 CellRank 转移核（PseudotimeKernel + ConnectivityKernel），
用 GPCCA 估计细胞的初始/终末状态与命运概率，识别两条谱系（C1_3、C2）的驱动基因（lineage drivers）。

**输入**:
- `output/object/seurat/se_trajectory_subset.h5ad`（脚本 03 产出的 AST 轨迹亚群, 含 counts_RNA 层）
- `output/object/slingshot/slingshot_pseudotime.csv` / `slingshot_weights.csv`（脚本 03 slingshot 产出）

**输出**（写入 `output/object/cellrank/`）:
- `C1_drivers.csv`、`C2_drivers.csv`（谱系驱动基因, 供脚本 03 第 5 章 TCGA 生存分析使用）
- `se_trajectory_subset.h5ad`（回写核与命运概率注释）
- 各随机游走/宏状态/命运概率/热图/基因趋势图（notebook 内联展示）

**上游脚本**: `03_trajectory_subset_analysis.R`（slingshot 章）
**下游脚本**: `03_trajectory_subset_analysis.R` 第 5 章（cellrank drivers → TCGA KM 曲线）

**环境依赖**: Python — cellrank, scanpy, scvelo, anndata, pandas, numpy;
`cr.models.GAMR` 需通过 rpy2 调用 R 的 mgcv 包（`pip install rpy2` + R 环境可用）。

> 注: 本 notebook 假定从仓库根目录（repository/）启动内核运行。

## 0. 环境与路径配置

In [ ]:
import os
import sys
import warnings

# 仓库根目录: 默认取 notebook 所在 scripts/ 目录的上一级;
# 若从其他位置启动内核, 请手动修改为仓库根目录的绝对路径
DIR_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DIR_OUT_CELLRANK = os.path.join(DIR_ROOT, "output", "object", "cellrank")
os.makedirs(DIR_OUT_CELLRANK, exist_ok=True)
os.chdir(DIR_OUT_CELLRANK)  # 后续相对路径输出(如 C1_drivers.csv)均落在此目录

import cellrank as cr
import scanpy as sc

sc.settings.set_figure_params(frameon=False, dpi=300)
cr.settings.verbosity = 2
warnings.simplefilter("ignore", category=UserWarning)

## 1. 读取数据与预处理
读取 AST 轨迹亚群 h5ad, 标准化 → 高变基因(3000) → PCA → KNN 图。

In [ ]:
adata = sc.read_h5ad(os.path.join(DIR_ROOT, "output", "object", "seurat", "se_trajectory_subset.h5ad"))
adata

In [ ]:
# 标准化 + log + 高变基因(使用 counts_RNA 层重新计算)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=3000)

In [ ]:
# 降维与近邻图(random_state=0 保证可重复)
sc.tl.pca(adata, random_state=0)
sc.pp.neighbors(adata, random_state=0)
sc.pl.embedding(adata, basis="X_umap", color=["res_0.2_clusters", "CytoTRACE2_Score"])

## 2. slingshot 伪时间整合（加权平均）
细胞可能同时位于多条谱系曲线上, 按 curve weights 对各谱系伪时间做加权平均,
得到单一伪时间变量 `slingshot_pseudotime_weighted`。

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad

pseudotime_matrix = pd.read_csv(
    os.path.join(DIR_ROOT, "output", "object", "slingshot", "slingshot_pseudotime.csv"), index_col=0)
curve_weights_matrix = pd.read_csv(
    os.path.join(DIR_ROOT, "output", "object", "slingshot", "slingshot_weights.csv"), index_col=0)

In [ ]:
# 对多谱系细胞按权重加权平均伪时间
adata.obs["slingshot_pseudotime_weighted"] = np.nan  # 初始化为 NaN
for i, cell_id in enumerate(adata.obs_names):
    cell_pseudotimes = pseudotime_matrix.loc[cell_id]
    cell_weights = curve_weights_matrix.loc[cell_id]

    # 过滤伪时间或权重缺失/权重为 0 的谱系
    valid_indices = (cell_pseudotimes.notna()) & (cell_weights.notna()) & (cell_weights > 0)

    if np.any(valid_indices):
        weighted_sum = np.sum(cell_pseudotimes[valid_indices] * cell_weights[valid_indices])
        sum_of_weights = np.sum(cell_weights[valid_indices])
        if sum_of_weights > 0:  # 避免除零
            adata.obs.loc[cell_id, "slingshot_pseudotime_weighted"] = weighted_sum / sum_of_weights

sc.pl.embedding(adata, basis="X_umap", color=["slingshot_pseudotime_weighted"])

## 3. 构建转移核
三种核: PseudotimeKernel(基于加权伪时间的偏向性随机游走)、CytoTRACEKernel(干性方向)、
ConnectivityKernel(KNN 转录组相似性)。最终组合核 = 0.8×Pseudotime + 0.2×Connectivity。

In [ ]:
from cellrank.kernels import PseudotimeKernel

# 3.1 伪时间核
pk = PseudotimeKernel(adata, time_key="slingshot_pseudotime_weighted")
pk.compute_transition_matrix()
pk.plot_random_walks(
    seed=0,
    n_sims=100,
    start_ixs={"seurat_clusters": "C0"},
    basis="X_umap",
    legend_loc="right",
    dpi=300,
)

In [ ]:
# 3.2 CytoTRACE 核: 需要 k-NN 插补数据, 借用 scVelo moments(以 X 同时充当 spliced/unspliced)
import scvelo as scv

if 'spliced' not in adata.layers or 'unspliced' not in adata.layers:
    adata.layers['spliced'] = adata.X
    adata.layers['unspliced'] = adata.X
scv.pp.moments(adata, method='sklearn')

ctk = cr.kernels.CytoTRACEKernel(adata)
ckt = ctk.compute_cytotrace().compute_transition_matrix()
ckt.plot_random_walks(
    seed=0,
    n_sims=100,
    start_ixs={"seurat_clusters": "C0"},
    basis="X_umap",
    legend_loc="right",
    dpi=300,
)

In [ ]:
# 3.3 连接性核
from cellrank.kernels import ConnectivityKernel

ck = ConnectivityKernel(adata).compute_transition_matrix()
ck.plot_random_walks(
    seed=0,
    n_sims=100,
    start_ixs={"seurat_clusters": "C0"},
    basis="X_umap",
    legend_loc="right",
    dpi=300,
)

In [ ]:
# 3.4 组合核: 伪时间核为主(0.8), 连接性核为辅(0.2)
combined_kernel = 0.8 * pk + 0.2 * ck
combined_kernel

## 4. GPCCA 估计宏状态与初始/终末状态
拟合并粗粒化为 10 个宏状态, 指定 2 个终末状态(C1_3、C2)与初始状态(C0_4)。

In [ ]:
from cellrank.estimators import GPCCA

g = GPCCA(combined_kernel)
g.fit(n_states=10, cluster_key="seurat_clusters")
g.plot_macrostates(which="all")
g.plot_coarse_T()

In [ ]:
# 预测终末状态(top 2)与初始状态, 并手动设定初始状态为 C0_4
g.predict_terminal_states(method="top_n", n_states=2)
g.plot_macrostates(which="terminal")
g.plot_macrostates(which="terminal", discrete=False)
g.predict_initial_states(n_states=1)
g.plot_macrostates(which="initial")
g.set_initial_states(states='C0_4')

## 5. 命运概率与谱系驱动基因
计算向各终末状态的命运概率; 识别 C1_3 与 C2 两条谱系的驱动基因并导出 CSV。

In [ ]:
# 5.1 命运概率
g.compute_fate_probabilities()
g.plot_fate_probabilities(legend_loc="right")
g.plot_fate_probabilities(same_plot=False)
g.plot_fate_probabilities(same_plot=True)

In [ ]:
# 5.2 谱系驱动基因(C1_3 / C2), 导出到当前工作目录(output/object/cellrank/)
C1_drivers = g.compute_lineage_drivers(lineages="C1_3")
C1_drivers.to_csv('C1_drivers.csv', index=True)

C2_drivers = g.compute_lineage_drivers(lineages="C2")
C2_drivers.to_csv('C2_drivers.csv', index=True)

top10_c1 = C1_drivers.head(10).index
print(top10_c1)
top10_c2 = C2_drivers.head(10).index
print(top10_c2)

In [ ]:
# 5.3 命运概率与 top 驱动基因在 UMAP 上的展示
adata.obs["fate_probabilities_C1"] = g.fate_probabilities["C1_3"].X.flatten()
sc.pl.embedding(
    adata,
    basis="umap",
    color=["fate_probabilities_C1"] + list(C1_drivers.index[:11]),
    color_map="viridis",
    s=50,
    ncols=3,
    vmax="p96",
)
adata.obs["fate_probabilities_C2"] = g.fate_probabilities["C2"].X.flatten()
sc.pl.embedding(
    adata,
    basis="umap",
    color=["fate_probabilities_C2"] + list(C2_drivers.index[:11]),
    color_map="viridis",
    s=50,
    ncols=3,
    vmax="p96",
)
sc.pl.umap(adata, color=['seurat_clusters', 'slingshot_pseudotime_weighted'])

## 6. 回写核与注释到 h5ad

In [ ]:
# 将各核的转移矩阵与参数写入 adata 并保存(输出到 output/object/cellrank/)
pk.write_to_adata()
ckt.write_to_adata()
ck.write_to_adata()
combined_kernel.write_to_adata()
adata.write("se_trajectory_subset.h5ad")

## 7. 表达趋势可视化（GAM 模型）
沿加权伪时间用 GAM 拟合基因表达趋势; 热图展示 top50 驱动基因的谱系特异性动力学。

In [ ]:
# top50 驱动基因(按相关性排序)
C1_top50_genes = C1_drivers.sort_values(by='C1_3_corr', ascending=False).head(50).index
C2_top50_genes = C2_drivers.sort_values(by='C2_corr', ascending=False).head(50).index

In [ ]:
# GAMR 模型经 rpy2 调用 R mgcv; 若缺少 rpy2 请先 pip install rpy2(需可用 R 环境)
model = cr.models.GAMR(adata)

# C1_3 谱系 top50 驱动基因热图(含命运概率条)
cr.pl.heatmap(
    adata,
    model=model,
    lineages="C1_3",
    cluster_key="seurat_clusters",
    show_fate_probabilities=True,
    genes=C1_top50_genes,
    time_key="slingshot_pseudotime_weighted",
    figsize=(12, 10),
    show_all_genes=True,
    weight_threshold=(1e-3, 1e-3),
)

In [ ]:
# C2 谱系 top50 驱动基因热图(基因聚类 + 树状图)
cr.pl.heatmap(
    adata,
    model=model,
    mode='lineages',
    cluster_genes=True,
    dendrogram=True,
    return_genes=True,
    return_figure=True,
    lineages="C2",
    cluster_key="seurat_clusters",
    show_fate_probabilities=True,
    genes=C2_top50_genes,
    time_key="slingshot_pseudotime_weighted",
    figsize=(12, 10),
    show_all_genes=True,
    weight_threshold=(1e-3, 1e-3),
)

In [ ]:
# 关键基因表达趋势: 干性/鳞状(Sox2, Trp63) vs 腺性(Napsa, Sftpc)
cr.pl.gene_trends(
    adata,
    model=model,
    data_key="X",
    genes=["Sox2", "Trp63", "Napsa", "Sftpc"],
    same_plot=True,
    ncols=2,
    time_key="slingshot_pseudotime_weighted",
    hide_cells=True,
)
cr.pl.heatmap(
    adata,
    model=model,
    data_key="X",
    genes=["Sox2", "Trp63", "Napsa", "Sftpc"],
    time_key="slingshot_pseudotime_weighted",
    cbar=False,
    show_all_genes=True,
)